# 02 · 동기식 환경 전이와 지연의 영향

이 노트북은 [GzDRL 논문](https://arxiv.org/html/2609.13243v1)의 식 (3), 실험 V-A(상태 전이 동기화)·V-B(훈련 재현성)를 이해하기 위한 **교육용 toy model**이다. 실제 Gazebo, ROS 2, PPO 훈련, 물리 로봇의 지연·처리량·재현성을 측정하거나 재현하지 않는다. 아래의 임의 지연은 우리가 주입한 값이며 논문에 보고된 ROS 2 측정치가 아니다. 논문 IV-A의 결정적 전이 보장은 **상태 관측**에 대한 것이며 렌더링 센서까지 보장한다고 확대하지 않는다. 외부 패키지는 NumPy만 사용한다.

논문의 순서는 정책 동작 $a_t$를 한 번 처리하고, 각 $k=1,\ldots,K$에서 **최신 상태**로 제어입력 $u_{t,k}$를 계산한 뒤 물리 단일 스텝을 실행하며, 마지막 PostUpdate 상태에서 관측·보상·종료를 만드는 것이다. 여기서는 1차원 위치/속도와 PD 제어기로 이 순서를 모사한다.


In [ ]:
import numpy as np

DT = 0.001
K = 10
KP, KD = 12.0, 4.0
ACTION_COUNT = 80
actions = 0.6 * np.sin(np.arange(ACTION_COUNT) * 0.13) + 0.2 * np.cos(np.arange(ACTION_COUNT) * 0.37)

def run_episode(action_sequence, action_delays, observation_delays):
    action_sequence = np.asarray(action_sequence, dtype=float)
    action_delays = np.asarray(action_delays, dtype=int)
    observation_delays = np.asarray(observation_delays, dtype=int)
    if not (len(action_sequence) == len(action_delays) == len(observation_delays)):
        raise ValueError('동작과 지연 배열 길이가 같아야 합니다.')
    if np.any(action_delays < 0) or np.any(action_delays >= K):
        raise ValueError('동작 지연은 0 이상 K 미만이어야 합니다.')
    if np.any(observation_delays < 0) or np.any(observation_delays >= K):
        raise ValueError('관측 지연은 0 이상 K 미만이어야 합니다.')
    x = v = active_target = 0.0
    history = [(x, v)]  # 각 물리 스텝의 PostUpdate 상태
    final_states, observations = [], []
    for requested, action_delay, observation_delay in zip(action_sequence, action_delays, observation_delays):
        processed_action = float(np.clip(requested, -1.0, 1.0))  # P_i(a_t): 정책 스텝당 한 번
        for k in range(1, K + 1):
            if k == 1 + action_delay:
                active_target = processed_action  # 다음 갱신 또는 주입한 지연 후 적용
            u = KP * (active_target - x) - KD * v  # C_i: 매번 최신 x,v로 계산
            v += DT * u                            # Run(1)의 1D 축소 모형
            x += DT * v
            history.append((x, v))                 # PostUpdate
        final_states.append(history[-1])
        observations.append(history[-1 - observation_delay])
    final_states = np.asarray(final_states)
    observations = np.asarray(observations)
    # 논문 V-A의 세 개념을 이산 물리 스텝 단위로 기록한다.
    metrics = {
        'action_to_update_steps': action_delays + 1,
        'observation_delay_steps': observation_delays.copy(),
        'intervening_physics_steps': action_delays.copy(),
    }
    return final_states, observations, metrics

zeros = np.zeros(ACTION_COUNT, dtype=int)
reference, reference_observations, sync_metrics = run_episode(actions, zeros, zeros)
assert np.array_equal(reference, reference_observations)
assert np.all(sync_metrics['action_to_update_steps'] == 1)
assert np.all(sync_metrics['observation_delay_steps'] == 0)
assert np.all(sync_metrics['intervening_physics_steps'] == 0)
print('동기식: 동작은 바로 다음 물리 갱신에 적용, 중간 물리 스텝 0개')


## 반복 실행과 V-A의 궤적 지표

동일한 초기 상태·동작·지연 배열이면 toy 전이가 완전히 같다. 다음에는 100회 실행마다 0~3개 물리 스텝의 동작·관측 지연을 무작위로 주입한다. 위치 spread는 같은 동작 인덱스에서 실행 간 위치의 표준편차, RMSE는 동기식 기준 궤적과의 제곱평균제곱근오차다. 이 비교는 **인위적 지연의 영향 시각화용 수치**이지 논문의 Gazebo/ROS 2 결과가 아니다.


In [ ]:
synchronous_repeats = np.stack([run_episode(actions, zeros, zeros)[0] for _ in range(100)])
assert np.array_equal(synchronous_repeats, np.broadcast_to(reference, synchronous_repeats.shape))

delayed_runs, observed_runs, delay_metrics = [], [], []
for trial in range(100):
    rng = np.random.default_rng(trial)
    action_delay = rng.integers(0, 4, size=ACTION_COUNT)
    observation_delay = rng.integers(0, 4, size=ACTION_COUNT)
    states, observed, metrics = run_episode(actions, action_delay, observation_delay)
    delayed_runs.append(states)
    observed_runs.append(observed)
    delay_metrics.append(metrics)
delayed_runs = np.stack(delayed_runs)
observed_runs = np.stack(observed_runs)
position_spread = np.std(delayed_runs[:, :, 0], axis=0, ddof=1)
trajectory_rmse = np.sqrt(np.mean((delayed_runs[:, :, 0] - reference[None, :, 0]) ** 2, axis=1))
assert delayed_runs.shape == (100, ACTION_COUNT, 2)
assert np.max(position_spread) > 0
assert np.mean(trajectory_rmse) > 0
assert np.any(observed_runs != delayed_runs)
assert np.array_equal(delayed_runs[17], run_episode(actions, delay_metrics[17]['intervening_physics_steps'], delay_metrics[17]['observation_delay_steps'])[0])
print(f'주입 지연 toy: 평균 위치 spread={np.mean(position_spread):.6f} m, 평균 궤적 RMSE={np.mean(trajectory_rmse):.6f} m')
print(f"평균 동작→갱신={np.mean([m['action_to_update_steps'].mean() for m in delay_metrics]):.2f} 물리 스텝")


## V-B의 seed 통제 원칙

논문 V-B는 별도 프로세스의 PPO 학습 5회씩을 같은 RL seed와 서로 다른 seed로 비교했다. 관측 정규화·환경 seed·평가 에피소드를 고정하고 checkpoint 매개변수 hash까지 비교했다. 여기에는 **학습기나 checkpoint가 없다**. 아래는 seed를 고정하면 무작위 지연 시퀀스와 궤적까지 재현된다는 좁은 원칙만 확인한다. 따라서 논문의 100% 동일 checkpoint 결과를 이 코드로 검증했다고 말할 수 없다.


In [ ]:
def seeded_run(seed):
    rng = np.random.default_rng(seed)
    action_delay = rng.integers(0, 4, size=ACTION_COUNT)
    observation_delay = rng.integers(0, 4, size=ACTION_COUNT)
    return run_episode(actions, action_delay, observation_delay)

same_a, same_obs_a, same_metric_a = seeded_run(42)
same_b, same_obs_b, same_metric_b = seeded_run(42)
other, _, _ = seeded_run(125)
assert np.array_equal(same_a, same_b)
assert np.array_equal(same_obs_a, same_obs_b)
assert all(np.array_equal(same_metric_a[key], same_metric_b[key]) for key in same_metric_a)
assert not np.array_equal(same_a, other)
print('동일 seed의 toy 궤적·관측·지연 지표가 모두 일치합니다.')
